# VAE Training

In [1]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from functools import partial

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vae").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vae.vae import VAE, vae_loss
from vae.vae_trainer import VAETrainingConfig, VAETrainer
from utils import NpyImageDataset, channel_normalize, channel_denormalize, get_device
from diffusers.optimization import get_cosine_schedule_with_warmup

DEVICE = get_device()
print(f'device: {DEVICE}')

device: mps


## Конфигурация

In [2]:
config = VAETrainingConfig(
    train_batch_size=16,
    eval_batch_size=8,
    num_epochs=50,
    learning_rate=1e-4,
    kl_weight=1e-3,
    sample_every_n_epochs=5,
    push_to_hub=False,
)

print(json.dumps(
    {k: str(v) if isinstance(v, tuple) else v for k, v in vars(config).items()
     if k not in ('channel_mean', 'channel_std')},
    indent=2
))

{
  "data_dir_train": "/Users/amir/sciml/sea_ice_data/train",
  "data_dir_valid": "/Users/amir/sciml/sea_ice_data/valid",
  "image_size": "(320, 256)",
  "num_workers_train": 6,
  "num_workers_val": 4,
  "train_batch_size": 16,
  "eval_batch_size": 8,
  "num_epochs": 50,
  "gradient_accumulation_steps": 1,
  "learning_rate": 0.0001,
  "lr_warmup_steps": 500,
  "mixed_precision": "no",
  "seed": 0,
  "kl_weight": 0.001,
  "sample_every_n_epochs": 5,
  "push_to_hub": false,
  "hub_model_id": "amirsadreev/vae_sea_ice",
  "base_output_dir": "checkpoints_vae",
  "resume_from_checkpoint": ""
}


## Данные

In [3]:
transform = partial(channel_normalize, channel_mean=config.channel_mean, channel_std=config.channel_std)

dataset_train = NpyImageDataset(folder=config.data_dir_train, transform=transform, mmap_mode='r')
dataset_valid = NpyImageDataset(folder=config.data_dir_valid, transform=transform, mmap_mode='r')

train_dataloader = torch.utils.data.DataLoader(
    dataset_train, batch_size=config.train_batch_size,
    shuffle=True, num_workers=config.num_workers_train,
    pin_memory=True, persistent_workers=True, prefetch_factor=2, drop_last=True,
)
valid_dataloader = torch.utils.data.DataLoader(
    dataset_valid, batch_size=config.eval_batch_size,
    shuffle=False, num_workers=config.num_workers_val,
    pin_memory=True, persistent_workers=True, prefetch_factor=2,
)

print(f'train: {len(dataset_train)}  valid: {len(dataset_valid)}')
print(f'train batches: {len(train_dataloader)}')

train: 255  valid: 255
train batches: 15


## Модель

In [4]:
model = VAE(
    in_channels=2,
    latent_channels=8,
    base_channels=64,
    scale_factor=8,
    max_channels=512,
)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

x_test = torch.randn(2, 2, 320, 256)
with torch.no_grad():
    recon, mu, logvar = model(x_test)
print(f'input:  {x_test.shape}')
print(f'latent: {mu.shape}')
print(f'output: {recon.shape}')

Parameters: 17,921,298
input:  torch.Size([2, 2, 320, 256])
latent: torch.Size([2, 8, 40, 32])
output: torch.Size([2, 2, 320, 256])


## Обучение

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=len(train_dataloader) * config.num_epochs,
)

trainer = VAETrainer(
    config=config,
    model=model,
    optimizer=optimizer,
    data_loader_train=train_dataloader,
    data_loader_val=valid_dataloader,
    lr_scheduler=lr_scheduler,
)

print(f'Output dir: {trainer.output_dir}')

Output dir: checkpoints_vae/run_20260313_144544


: 

In [ ]:
trainer.train_loop()

  0%|          | 0/15 [00:00<?, ?it/s]

/Users/amir/sciml/diffusion_data_assimilation/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

...

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

## Кривая потерь

In [1]:
history = trainer.val_history
epochs     = [h['epoch']    for h in history]
val_loss   = [h['val_loss'] for h in history]
val_recon  = [h['val_recon'] for h in history]
val_kl     = [h['val_kl']   for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, values, title in zip(axes,
        [val_loss, val_recon, val_kl],
        ['Total loss', 'Recon loss', 'KL']):
    ax.plot(epochs, values)
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.grid(True)
plt.tight_layout()
plt.show()

NameError: name 'trainer' is not defined

## Загрузка лучшей модели и визуализация реконструкций

In [ ]:
best_path = os.path.join(trainer.output_dir, 'best_model.pth')
model.load_state_dict(torch.load(best_path, map_location='cpu'))
model.eval().to(DEVICE)
print(f'Loaded: {best_path}')

In [ ]:
N = 8

indices = np.linspace(0, len(dataset_valid) - 1, N, dtype=int)
batch = torch.stack([dataset_valid[i] for i in indices]).to(DEVICE)

with torch.no_grad():
    recon, mu, logvar = model(batch)

batch_dn = channel_denormalize(batch.cpu().clone(), config.channel_mean, config.channel_std)
recon_dn = channel_denormalize(recon.cpu().clone(), config.channel_mean, config.channel_std)

fig, axes = plt.subplots(2, N, figsize=(N * 3, 6))
for i in range(N):
    axes[0, i].imshow(batch_dn[i, 0].numpy(), cmap='Blues_r')
    axes[0, i].set_title(f'Truth #{indices[i]}')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon_dn[i, 0].numpy(), cmap='Blues_r')
    axes[1, i].set_title(f'Recon #{indices[i]}')
    axes[1, i].axis('off')
plt.suptitle('Concentration (channel 0)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, N, figsize=(N * 3, 6))
for i in range(N):
    axes[0, i].imshow(batch_dn[i, 1].numpy(), cmap='viridis')
    axes[0, i].set_title(f'Truth #{indices[i]}')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon_dn[i, 1].numpy(), cmap='viridis')
    axes[1, i].set_title(f'Recon #{indices[i]}')
    axes[1, i].axis('off')
plt.suptitle('Thickness (channel 1)')
plt.tight_layout()
plt.show()

## MSE реконструкции на val

In [ ]:
total_mse = torch.zeros(2)
n_batches = 0

model.eval()
with torch.no_grad():
    for batch in valid_dataloader:
        batch = batch.to(DEVICE)
        recon, _, _ = model(batch)
        mse = ((batch - recon) ** 2).mean(dim=(0, 2, 3)).cpu()
        total_mse += mse
        n_batches += 1

mean_mse = total_mse / n_batches
print(f'Recon MSE (normalized):')
print(f'  Concentration: {mean_mse[0]:.6f}')
print(f'  Thickness:     {mean_mse[1]:.6f}')

## Латентное пространство — распределение

In [ ]:
all_mu     = []
all_logvar = []

model.eval()
with torch.no_grad():
    for batch in valid_dataloader:
        mu, logvar = model.encode(batch.to(DEVICE))
        all_mu.append(mu.cpu())
        all_logvar.append(logvar.cpu())

all_mu     = torch.cat(all_mu)
all_logvar = torch.cat(all_logvar)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_mu.flatten().numpy(), bins=100)
axes[0].set_title('mu distribution')
axes[0].set_xlabel('value')
axes[1].hist(all_logvar.flatten().numpy(), bins=100)
axes[1].set_title('logvar distribution')
axes[1].set_xlabel('value')
plt.tight_layout()
plt.show()

print(f'mu:     mean={all_mu.mean():.4f}  std={all_mu.std():.4f}')
print(f'logvar: mean={all_logvar.mean():.4f}  std={all_logvar.std():.4f}')